In [1]:
# @title Error Handling
!nvidia-smi

Thu Jun 19 10:33:12 2025       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 550.54.15              Driver Version: 550.54.15      CUDA Version: 12.4     |
|-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  NVIDIA A100-SXM4-40GB          Off |   00000000:00:04.0 Off |                    0 |
| N/A   32C    P0             48W /  400W |       0MiB /  40960MiB |      0%      Default |
|                                         |                        |             Disabled |
+-----------------------------------------+-----

In [2]:
# @title Plotly Installation
! pip install plotly -q

In [3]:
# @title Git clone of Point-E Github Respiratory
!git clone https://github.com/openai/point-e

Cloning into 'point-e'...
remote: Enumerating objects: 57, done.
remote: Counting objects: 100% (30/30), done.
remote: Compressing objects: 100% (28/28), done.
remote: Total 57 (delta 4), reused 2 (delta 2), pack-reused 27 (from 1)
Receiving objects: 100% (57/57), 1.57 MiB | 51.73 MiB/s, done.
Resolving deltas: 100% (7/7), done.


In [4]:
# @title Change of Current Directory
%cd point-e

/content/point-e


In [5]:
# @title Installation of Point-E Package
! pip install -e .

Obtaining file:///content/point-e
  Preparing metadata (setup.py) ... done
  Cloning https://github.com/openai/CLIP.git to /tmp/pip-install-bai3ulru/clip_0a18283a851745028e5bbae46316ded4
  Running command git clone --filter=blob:none --quiet https://github.com/openai/CLIP.git /tmp/pip-install-bai3ulru/clip_0a18283a851745028e5bbae46316ded4
  Resolved https://github.com/openai/CLIP.git to commit dcba3cb2e2827b402d2701e7e1c7d9fed8a20ef1
  Preparing metadata (setup.py) ... done
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 87.2/87.2 kB 7.6 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 363.4/363.4 MB 3.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 13.8/13.8 MB 109.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 24.6/24.6 MB 96.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 883.7/883.7 kB 57.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 664.8/664.8 MB 1.7 MB/s eta 0:00:00


In [6]:
# @title Imports
import torch
from tqdm.auto import tqdm

from point_e.diffusion.configs import DIFFUSION_CONFIGS, diffusion_from_config
from point_e.diffusion.sampler import PointCloudSampler
from point_e.models.download import load_checkpoint
from point_e.models.configs import MODEL_CONFIGS, model_from_config
from point_e.util.plotting import plot_point_cloud

In [ ]:
# @title Creation of Base Model
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

print('creating base model...')
base_name = 'base40M-textvec'
base_model = model_from_config(MODEL_CONFIGS[base_name], device)
base_model.eval()
base_diffusion = diffusion_from_config(DIFFUSION_CONFIGS[base_name])

print('creating upsample model...')
upsampler_model = model_from_config(MODEL_CONFIGS['upsample'], device)
upsampler_model.eval()
upsampler_diffusion = diffusion_from_config(DIFFUSION_CONFIGS['upsample'])

print('downloading base checkpoint...')
base_model.load_state_dict(load_checkpoint(base_name, device))

print('downloading upsampler checkpoint...')
upsampler_model.load_state_dict(load_checkpoint('upsample', device))

creating base model...


100%|████████████████████████████████████████| 890M/890M [00:07<00:00, 124MiB/s]


creating upsample model...
downloading base checkpoint...


  0%|          | 0.00/161M [00:00<?, ?iB/s]

downloading upsampler checkpoint...


  0%|          | 0.00/162M [00:00<?, ?iB/s]

In [ ]:
# @title Sampler
sampler = PointCloudSampler(
    device=device,
    models=[base_model, upsampler_model],
    diffusions=[base_diffusion, upsampler_diffusion],
    num_points=[1024, 4096 - 1024],
    aux_channels=['R', 'G', 'B'],
    guidance_scale=[3.0, 0.0],
    model_kwargs_key_filter=('texts', ''), # Do not condition the upsampler at all
)

In [ ]:
# @title Prompt
# Set a prompt to condition on.
prompt = 'An Apple'

# Produce a sample from the model.
samples = None
for x in tqdm(sampler.sample_batch_progressive(batch_size=1, model_kwargs=dict(texts=[prompt]))):
    samples = x

In [ ]:
# @title Sampler Output
pc = sampler.output_to_point_clouds(samples)[0]

In [ ]:
# @title Point Cloud Plot
fig = plot_point_cloud(pc, grid_size=3, fixed_bounds=((-0.75, -0.75, -0.75),(0.75, 0.75, 0.75)))

In [ ]:
# @title Graphical Plot
import plotly.graph_objects as go

In [ ]:
# @title Graphical Plot Computation
fig_plotly = go.Figure(
        data=[
            go.Scatter3d(
                x=pc.coords[:,0], y=pc.coords[:,1], z=pc.coords[:,2],
                mode='markers',
                marker=dict(
                  size=2,
                  color=['rgb({},{},{})'.format(r,g,b) for r,g,b in zip(pc.channels["R"], pc.channels["G"], pc.channels["B"])],
              )
            )
        ],
        layout=dict(
            scene=dict(
                xaxis=dict(visible=False),
                yaxis=dict(visible=False),
                zaxis=dict(visible=False)
            )
        ),
    )

In [ ]:
# @title Plot Presented
fig_plotly.show(renderer="colab")

In [ ]:
# @title Open3D Installation
!pip install open3d # Install the open3d library
import open3d as o3d

# Assuming you have the 3D point cloud as a numpy array, convert it to a mesh
# Example: If you already have vertices for the apple, stem, etc., use them to create a mesh
# Combine the apple, stem, and leaf into a single mesh
apple_mesh = o3d.geometry.TriangleMesh.create_sphere(radius=1.0)
apple_mesh.paint_uniform_color([1.0, 0.0, 0.0])  # Red apple color

stem_mesh = o3d.geometry.TriangleMesh.create_cylinder(radius=0.1, height=0.5)
stem_mesh.paint_uniform_color([0.0, 1.0, 0.0])  # Green stem
stem_mesh.translate([0, 0, 1.0])  # Position the stem on top of the apple

# Combine the meshes into a single mesh
combined_mesh = apple_mesh + stem_mesh

# Save the combined mesh as an OBJ file
o3d.io.write_triangle_mesh("apple_with_stem.obj", combined_mesh)
print("Model saved as 'apple_with_stem.obj'")

In [ ]:
# @title OBJ Computation
from google.colab import files
import open3d as o3d

# Assuming you have the 3D point cloud as a numpy array, convert it to a mesh
# Example: If you already have vertices for the apple, stem, etc., use them to create a mesh
# Combine the apple, stem, and leaf into a single mesh
apple_mesh = o3d.geometry.TriangleMesh.create_sphere(radius=1.0)
apple_mesh.paint_uniform_color([1.0, 0.0, 0.0])  # Red apple color

stem_mesh = o3d.geometry.TriangleMesh.create_cylinder(radius=0.1, height=0.5)
stem_mesh.paint_uniform_color([0.0, 1.0, 0.0])  # Green stem
stem_mesh.translate([0, 0, 1.0])  # Position the stem on top of the apple

# Combine the meshes into a single mesh
combined_mesh = apple_mesh + stem_mesh

# Save the combined mesh as an OBJ file
o3d.io.write_triangle_mesh("apple_with_stem.obj", combined_mesh)
print("Model saved as 'apple_with_stem.obj'")

# Download the saved OBJ file immediately after creation
files.download("apple_with_stem.obj")